# Pre-Refusal Geometry Atlas: Full Paper Experiments

This notebook runs the stronger v2 evaluation suite on Colab/T4:

- Qwen2.5 hidden-state extraction on counterfactual/hard-negative prompts
- layer-wise probes
- text/control baselines
- family-heldout generalization
- prefix emergence heatmap
- harmful-intent direction geometry
- logit-lens direction analysis
- paper-style summary figure

At the end it downloads `pre_refusal_paper_results.zip`.

In [ ]:
import os, sys, platform, subprocess, json, shutil
from pathlib import Path

print('Python:', sys.version)
print('Platform:', platform.platform())
import torch
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Switch Colab to Runtime -> Change runtime type -> T4 GPU, restart, and run again.')
print('GPU:', torch.cuda.get_device_name(0))

## Clone the repository

In [ ]:
REPO_URL = 'https://github.com/DaviBonetto/algoverse.git'
PROJECT_DIR = Path('/content/pre-refusal-signatures')

if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)
subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_DIR)], check=True)
os.chdir(PROJECT_DIR)
print('cwd:', Path.cwd())
subprocess.run(['git', 'log', '--oneline', '-3'], check=False)

## Install dependencies

In [ ]:
!python -m pip install -q --upgrade transformers accelerate scikit-learn numpy pandas matplotlib seaborn pyyaml tqdm pytest

## Configure paper run

Defaults are T4-safe. If the run succeeds and you want the heaviest version, set `MAX_LENGTH = 512` and re-run extraction.

In [ ]:
import yaml

MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'
DATA_PATH = 'data/prompts_v2.jsonl'
CONFIG_PATH = 'configs/paper_t4.yaml'
DEVICE = 'cuda'
MAX_PROMPTS = None
MAX_LENGTH = 256
RUN_PREFIX = True
PREFIX_FRACTIONS = [0.25, 0.5, 0.75, 1.0]

config = yaml.safe_load(Path(CONFIG_PATH).read_text())
config['model_name'] = MODEL_NAME
config['data_path'] = DATA_PATH
config['device'] = DEVICE
config['max_length'] = MAX_LENGTH
Path(CONFIG_PATH).write_text(yaml.safe_dump(config, sort_keys=False))
print(Path(CONFIG_PATH).read_text())

## Validate v2 dataset

In [ ]:
!python scripts/00_validate_dataset.py --data data/prompts_v2.jsonl --min-per-label 20

## Extract Qwen hidden states

In [ ]:
import shlex
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
cmd = ['python', 'scripts/01_extract_hidden_states.py', '--config', CONFIG_PATH, '--device', DEVICE]
if MAX_PROMPTS is not None:
    cmd += ['--max-prompts', str(MAX_PROMPTS)]
print('Running:', ' '.join(shlex.quote(x) for x in cmd))
result = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(result.stdout)
if result.returncode != 0:
    raise RuntimeError(f'extraction failed: {result.returncode}')

## Main probes and figures

In [ ]:
!python scripts/02_train_layer_probes.py --states outputs/hidden_states.npz
!python scripts/03_make_figures.py --states outputs/hidden_states.npz --metrics reports/layer_probe_metrics.csv

## Paper-grade controls and geometry

In [ ]:
!python scripts/05_run_baselines.py --states outputs/hidden_states.npz
!python scripts/06_group_heldout_eval.py --states outputs/hidden_states.npz
!python scripts/08_direction_geometry.py --states outputs/hidden_states.npz
!python scripts/09_logit_lens_direction.py --states outputs/hidden_states.npz --device cpu

## Prefix emergence map

This is the slowest extra experiment because it extracts hidden states for multiple prompt-prefix fractions.

In [ ]:
if RUN_PREFIX:
    cmd = ['python', 'scripts/07_prefix_emergence.py', '--config', CONFIG_PATH, '--data', DATA_PATH, '--device', DEVICE]
    if MAX_PROMPTS is not None:
        cmd += ['--max-prompts', str(MAX_PROMPTS)]
    cmd += ['--fractions'] + [str(x) for x in PREFIX_FRACTIONS]
    print('Running:', ' '.join(shlex.quote(x) for x in cmd))
    result = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout)
    if result.returncode != 0:
        raise RuntimeError(f'prefix emergence failed: {result.returncode}')
else:
    print('Skipping prefix emergence.')

## Build paper summary figure and run tests

In [ ]:
!python scripts/10_make_paper_figure.py --reports-dir reports --figures-dir figures
!pytest -q

## Inspect key outputs

In [ ]:
import pandas as pd
from IPython.display import display, Image

for path in ['reports/layer_probe_metrics.csv', 'reports/baseline_comparison.csv', 'reports/family_heldout_results.csv', 'reports/direction_geometry.csv']:
    print('\n', path)
    display(pd.read_csv(path).head())

for image in ['paper_summary_figure.png', 'prefix_emergence_heatmap.png', 'direction_geometry.png', 'baseline_comparison.png']:
    path = Path('figures') / image
    if path.exists():
        print(path)
        display(Image(filename=str(path)))

## Save run metadata

In [ ]:
import datetime
metadata = {
    'timestamp_utc': datetime.datetime.utcnow().isoformat() + 'Z',
    'model_name': MODEL_NAME,
    'data_path': DATA_PATH,
    'config_path': CONFIG_PATH,
    'max_prompts': MAX_PROMPTS,
    'max_length': MAX_LENGTH,
    'run_prefix': RUN_PREFIX,
    'prefix_fractions': PREFIX_FRACTIONS,
    'python': sys.version,
    'platform': platform.platform(),
    'torch': torch.__version__,
    'cuda_available': torch.cuda.is_available(),
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}
try:
    metadata['git_commit'] = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
except Exception:
    metadata['git_commit'] = None
Path('reports/paper_run_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
print(json.dumps(metadata, indent=2))

## Package results

In [ ]:
import zipfile
zip_path = Path('/content/pre_refusal_paper_results.zip')
include_dirs = ['figures', 'reports', 'outputs', 'configs', 'data']
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for folder in include_dirs:
        for path in Path(folder).rglob('*'):
            if path.is_file():
                zf.write(path, arcname=str(path))
    for file_name in ['README.md', 'requirements.txt', 'pyproject.toml']:
        if Path(file_name).exists():
            zf.write(file_name)
print('Created:', zip_path)
print('Size MB:', round(zip_path.stat().st_size / 1_000_000, 2))

In [ ]:
from google.colab import files
files.download('/content/pre_refusal_paper_results.zip')